# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
aviation_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")

print(f"Inspect NaNs:\n{aviation_df.isna().sum()}")
print(f"Inspect datatypes:\n{aviation_df.dtypes}")
print(f"Summary statistics:\n{aviation_df.describe()}")

Inspect NaNs:
Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          

/var/folders/y_/8lgqgn_17j53v04m52jxzw7c0000gn/T/ipykernel_60161/1469194797.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
print(f"Aircraft categories:{aviation_df["Aircraft.Category"].unique()}")
print(f"Amateur built:{aviation_df["Amateur.Built"].unique()}")

aviation_df = aviation_df[aviation_df["Aircraft.Category"] == "Airplane"]
aviation_df = aviation_df[aviation_df["Amateur.Built"] == "No"]
aviation_df = aviation_df[pd.to_datetime(aviation_df["Event.Date"]).dt.year >= 1983]

aviation_df.loc[:, ["Aircraft.Category", "Amateur.Built", "Event.Date"]].head()

Aircraft categories:[nan 'Airplane' 'Helicopter' 'Glider' 'Balloon' 'Gyrocraft' 'Ultralight'
 'Unknown' 'Blimp' 'Powered-Lift' 'Weight-Shift' 'Powered Parachute'
 'Rocket' 'WSFT' 'UNK' 'ULTR']
Amateur built:['No' 'Yes' nan]


,Aircraft.Category,Amateur.Built,Event.Date
4149,Airplane,No,1983-03-18
4150,Airplane,No,1983-03-18
4171,Airplane,No,1983-03-20
4285,Airplane,No,1983-04-02
5957,Airplane,No,1983-08-21


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
#Injury.Severity records either fatal or non-fatal, but does not count serious injuries, so the Total.Fatal.Injuries and Total.Serious.Injuries must be added manually.

# Clean passenger condition data, then sum passenger conditions across the columns and store to "Passengers" column.
aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"].fillna(0)
aviation_df["Number.of.Passengers"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"].sum(axis=1)
# Severity.Likelihood is a value [0:1]: sum of fatal and serious injuries divided by number of passengers.
aviation_df["Severity.Likelihood"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Serious.Injuries"].sum(axis=1) / aviation_df["Number.of.Passengers"]
aviation_df["Severity.Likelihood"] = aviation_df["Severity.Likelihood"].fillna(0)

aviation_df.loc[:, ["Total.Serious.Injuries", "Total.Fatal.Injuries", "Number.of.Passengers","Severity.Likelihood"]].head()

,Total.Serious.Injuries,Total.Fatal.Injuries,Number.of.Passengers,Severity.Likelihood
4149,0.0,0.0,588.0,0.0
4150,0.0,0.0,588.0,0.0
4171,1.0,1.0,2.0,1.0
4285,0.0,1.0,5.0,0.2
5957,0.0,0.0,289.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print(f"Aircraft damage:{aviation_df["Aircraft.damage"].unique()}")

aviation_df["Aircraft.damage"] = aviation_df["Aircraft.damage"].fillna("Unknown")
aviation_df["Aircraft.Destroyed"] = aviation_df["Aircraft.damage"] == "Destroyed"

aviation_df.loc[:, ["Aircraft.damage", "Aircraft.Destroyed"]]

Aircraft damage:['Minor' 'Destroyed' nan 'Substantial' 'Unknown']


,Aircraft.damage,Aircraft.Destroyed
4149,Minor,False
4150,Minor,False
4171,Destroyed,True
4285,Unknown,False
5957,Minor,False
...,...,...
88869,Substantial,False
88873,Substantial,False
88876,Substantial,False
88877,Substantial,False


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
## Drop NaN Make
aviation_df = aviation_df.dropna(subset="Make")

## Make each Make in UPPERCASE
aviation_df["Make"] = aviation_df["Make"].str.upper()

# Remove periods and commas
aviation_df["Make"] = aviation_df["Make"].str.replace('.', '')
aviation_df["Make"] = aviation_df["Make"].str.replace(',', '')

# Remove the suffix of each Make, as these are inconsistent in the dataset,
# and can cause same Make to be considered different
# Use regex $ so INC, LTD, etc. is only removed if at the end
aviation_df["Make"] = aviation_df["Make"].str.replace(r'\s*(INC|LTD|LLC|CORP|CORPORATION|AVIATION|COMPANY|CO|INCORPORATE|INCORPORATED)\.?$', '', regex=True)

# Strip whitespace from beginning and end of Make
aviation_df["Make"] = aviation_df["Make"].str.strip()

# Keep the Makes that have atleast 50 entries
make_counts = aviation_df["Make"].value_counts()
aviation_df = aviation_df[aviation_df["Make"].map(make_counts) >= 50]

aviation_df.loc[:, "Make"].head()

4150          BOEING
4171           PIPER
4285    DE HAVILLAND
6760          BOEING
6806           BEECH
Name: Make, dtype: object

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
aviation_df = aviation_df.dropna(subset='Model')

print(aviation_df["Model"].value_counts().head())
print(aviation_df["Make"].value_counts().head())

# Unique CESSNA Models (Models are NOT unique to each Make)
print(aviation_df.loc[aviation_df["Make"] == "CESSNA", "Model"].unique())

aviation_df["Aircraft.Type"] = aviation_df["Make"] + ' ' + aviation_df["Model"]
aviation_df.loc[:, ["Make", "Model", "Aircraft.Type"]].head()

Model
172     769
737     403
152     316
182     304
172S    276
Name: count, dtype: int64
Make
CESSNA         7142
PIPER          3988
BEECH          1431
BOEING         1270
AIR TRACTOR     432
Name: count, dtype: int64
['180K' 'R172E' '182A' '182Q' 'T303' '152' '441' '208' 'TU206F' '150K'
 '310R' '210L' '182B' '182' '208B' '170B' '172' '150' '185B' '172N' '120'
 'T210' 'K172R' '185' '550' '337' 'P206C' '414' '205' 'TU-206G' '182C'
 '182E' '650-0220' 'U206G' '172M' '172P' '182M' '402C' '182N' '206'
 'T310R' '340A' '525' 'P206' '210F' 'TR182' 'C-208 Caravan' '182J' '182S'
 'TU206G' '337D' 'A188' '182L' 'U206' '188' '150H' '188B' 'T210M' 'T210N'
 'A185F' '150G' '182F' '180' 'A150M' '182R' '411' 'R172HP' '172S' '190'
 '182P' '177' '182H' '172K' 'TU-206F' 'R172K' '140' '185F' '172RG' '210D'
 '150C' '195' '172E' 'C-170' '150M' '402B' '182K' '150F' '180 H' 'U-206G'
 'U206F' '172R' '172F' '150L' '177A' 'C195A' '172C' '172A' '150J' '182T'
 'C-180' '180A' 'A185E' 'C-152' '305 A' '150E' '180J

,Make,Model,Aircraft.Type
4150,BOEING,747,BOEING 747
4171,PIPER,PA-28-140,PIPER PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND DHC-6
6760,BOEING,727-200,BOEING 727-200
6806,BEECH,C35,BEECH C35


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
aviation_df["Engine.Type"] = aviation_df["Engine.Type"].replace("UNK", "Unknown")
print(aviation_df["Engine.Type"].unique())

aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].str.upper()
aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].replace("UNK", "Unknown")
print(aviation_df["Weather.Condition"].unique())

# Number.of.Engines is clean
print(aviation_df["Number.of.Engines"].unique())

aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].replace("Air Race show", "Air Race/show")
print(aviation_df["Purpose.of.flight"].unique())

# Broad.phase.of.flight is clean
print(aviation_df["Broad.phase.of.flight"].unique())

aviation_df.loc[:, ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]]

['Turbo Fan' 'Reciprocating' 'Turbo Prop' 'Turbo Jet' nan 'Unknown'
 'Turbo Shaft' 'Geared Turbofan']
['VMC' 'IMC' 'Unknown' nan]
[ 4.  1.  2.  3. nan  0.]
[nan 'Personal' 'Skydiving' 'Unknown' 'Aerial Application' 'Positioning'
 'Instructional' 'Business' 'Public Aircraft' 'Ferry' 'Other Work Use'
 'Aerial Observation' 'Executive/corporate' 'Public Aircraft - Federal'
 'Air Race/show' 'Flight Test' 'Public Aircraft - State' 'Glider Tow'
 'Banner Tow' 'Firefighting' 'Public Aircraft - Local' 'Air Drop' 'PUBS'
 'ASHO']
['Taxi' 'Cruise' 'Standing' 'Climb' 'Takeoff' 'Landing' 'Approach'
 'Maneuvering' 'Descent' nan 'Go-around' 'Unknown' 'Other']


,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
4150,Turbo Fan,VMC,4.0,NaN,Taxi
4171,Reciprocating,IMC,1.0,Personal,Cruise
4285,Turbo Prop,VMC,2.0,Skydiving,Standing
6760,Turbo Fan,VMC,3.0,Unknown,Taxi
6806,Reciprocating,VMC,1.0,Personal,Climb
...,...,...,...,...,...
88865,NaN,VMC,1.0,Instructional,NaN
88869,NaN,NaN,2.0,NaN,NaN
88873,NaN,VMC,1.0,Personal,NaN
88877,NaN,VMC,1.0,Personal,NaN


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
print(f"All Columns: {aviation_df.columns}")

aviation_df = aviation_df.dropna(thresh=len(aviation_df) * 0.5, axis=1)

print(f"New Columns: {aviation_df.columns}")

print(aviation_df.shape)

All Columns: Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date', 'Number.of.Passengers', 'Severity.Likelihood',
       'Aircraft.Destroyed', 'Aircraft.Type'],
      dtype='object')
New Columns: Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Numbe

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
aviation_df.to_csv('Aviation_Cleaned.csv', index=False)